# ALE 3D CNN NeuroVLM Experiment

Runs both ALE3DCNN variants explicitly: DiFuMo-compatible and atlas-free. Outputs are written to Google Drive, while packed caches are copied to local Colab disk for faster training.

## 1. Runtime / GPU Check

In [ ]:
import os, platform, subprocess, sys, time, shutil, json
print('Python:', sys.version)
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)
print('Platform:', platform.platform())

## 2. Dependency Install

In [ ]:
# PyG is not required for the dense ALE CNN path.
!pip -q install nilearn nibabel huggingface-hub safetensors adapters transformers pyarrow matplotlib pandas scikit-learn tqdm umap-learn

## 3. Repo Clone / Pull

In [ ]:
REPO_URL = 'https://github.com/neurovlm/neurovlm.git'
REPO_BRANCH = 'neurovlm_gnn'
REPO_DIR = '/content/neurovlm_gnn'
if not os.path.exists(REPO_DIR):
    !git clone --branch $REPO_BRANCH --single-branch $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin $REPO_BRANCH
    !git -C $REPO_DIR checkout $REPO_BRANCH
    !git -C $REPO_DIR pull --ff-only origin $REPO_BRANCH
os.chdir(REPO_DIR)
!pip -q install -e '.[viz,notebook,metrics]'
print('Working directory:', os.getcwd())

## 4. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Shared Config

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/neurovlm_ale3dcnn'
DRIVE_CACHE_DIR = f'{DRIVE_ROOT}/data/ale_caches'
LOCAL_CACHE_DIR = '/content/ale_caches'
RUN_NAME_OR_TIMESTAMP = time.strftime('%Y%m%d_%H%M%S')

EPOCHS = 200
BATCH_SIZE = 16
RESOLUTION_MM = 4
KERNEL_FWHM_MM = 9
BASE_CHANNELS = 16
NUM_BLOCKS = 3
OUT_DIM = 384
DROPOUT = 0.1
VAL_INTERVAL = 5
EARLY_STOPPING_PATIENCE = 25
MONITOR_METRIC = 'paper_recall_curve_auc'
CACHE_DTYPE = 'float16'
NUM_WORKERS = 2
COMPARISON_FILE = f'{DRIVE_ROOT}/runs/ale_model_comparison.csv'

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/runs', exist_ok=True)

SHARED_CONFIG = {
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'resolution_mm': RESOLUTION_MM,
    'kernel_fwhm_mm': KERNEL_FWHM_MM,
    'base_channels': BASE_CHANNELS,
    'num_blocks': NUM_BLOCKS,
    'out_dim': OUT_DIM,
    'dropout': DROPOUT,
    'val_interval': VAL_INTERVAL,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'monitor_metric': MONITOR_METRIC,
    'cache_dtype': CACHE_DTYPE,
}
with open(f'{DRIVE_ROOT}/shared_training_config.json', 'w') as f:
    json.dump(SHARED_CONFIG, f, indent=2)
print(json.dumps(SHARED_CONFIG, indent=2))
print('Drive cache directory:', DRIVE_CACHE_DIR)
print('Expected DiFuMo cache:', f'{DRIVE_CACHE_DIR}/difumo_compatible_{RESOLUTION_MM}mm_fwhm{KERNEL_FWHM_MM}_crop_{CACHE_DTYPE}.pt')
print('Expected atlas-free cache:', f'{DRIVE_CACHE_DIR}/atlas_free_{RESOLUTION_MM}mm_fwhm{KERNEL_FWHM_MM}_crop_{CACHE_DTYPE}.pt')

## 6. Helper Function

In [ ]:
COMPLETED_RUNS = []

import argparse
import importlib.util

TRAIN_MODULE_PATH = os.path.join(REPO_DIR, 'experiments', 'train_ale_cnn.py')
_spec = importlib.util.spec_from_file_location('train_ale_cnn_colab', TRAIN_MODULE_PATH)
train_ale_cnn = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(train_ale_cnn)

def _cache_name(mode):
    return f'{mode}_{RESOLUTION_MM}mm_fwhm{KERNEL_FWHM_MM}_crop_{CACHE_DTYPE}.pt'

def run_ale_experiment(mode, model, run_name):
    drive_run_dir = f'{DRIVE_ROOT}/runs/{model}_{mode}_{run_name}'
    local_cache_file = f'{LOCAL_CACHE_DIR}/{_cache_name(mode)}'
    drive_cache_file = f'{DRIVE_CACHE_DIR}/{_cache_name(mode)}'
    os.makedirs(drive_run_dir, exist_ok=True)
    os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)

    local_cache_existed = os.path.exists(local_cache_file)
    print(f'Expected Drive cache: {drive_cache_file}', flush=True)
    if os.path.exists(drive_cache_file):
        print(f'Copying Drive cache to local disk: {drive_cache_file} -> {local_cache_file}', flush=True)
        shutil.copy2(drive_cache_file, local_cache_file)
    elif os.path.exists(local_cache_file):
        print(f'No Drive cache found; using existing local cache: {local_cache_file}', flush=True)
    else:
        print(f'No Drive/local cache found; training script will build local packed cache: {local_cache_file}', flush=True)

    args = argparse.Namespace(
        mode=mode,
        model=model,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        batch_size_auto=False,
        lr_cnn=1e-4,
        lr_proj=1e-5,
        warmup_epochs=5,
        temperature=0.07,
        val_interval=VAL_INTERVAL,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        monitor_metric=MONITOR_METRIC,
        base_channels=BASE_CHANNELS,
        num_blocks=NUM_BLOCKS,
        out_dim=OUT_DIM,
        dropout=DROPOUT,
        norm='group',
        pooling='max',
        mlp_hidden_dim=1024,
        kernel_fwhm_mm=KERNEL_FWHM_MM,
        resolution_mm=RESOLUTION_MM,
        crop_to_brain=True,
        normalize='max',
        no_clamp=False,
        cache_dtype=CACHE_DTYPE,
        cache_file=local_cache_file,
        force_rebuild_cache=False,
        build_cache_only=False,
        max_papers=None,
        text_proj_init='random',
        freeze_text_proj=False,
        device='cuda' if __import__('torch').cuda.is_available() else 'auto',
        amp=True,
        num_workers=NUM_WORKERS,
        pin_memory=__import__('torch').cuda.is_available(),
        seed=42,
        val_frac=0.1,
        test_frac=0.1,
        checkpoint_dir=f'{drive_run_dir}/checkpoints',
        run_dir=drive_run_dir,
        comparison_file=COMPARISON_FILE,
        save_plots=True,
        umap=True,
        eval_neurovlm_baseline=False,
    )

    print('Starting in-notebook training with args:', flush=True)
    print(json.dumps(vars(args), indent=2), flush=True)

    run_dir = Path(args.run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    Path(args.checkpoint_dir).mkdir(parents=True, exist_ok=True)
    __import__('torch').manual_seed(args.seed)
    __import__('numpy').random.seed(args.seed)

    ds, train_ds, val_ds, test_ds, payload, preprocess_config = train_ale_cnn.build_dataset(args)
    brain_encoder, text_proj = train_ale_cnn.build_model(args, ds.input_shape)
    device = train_ale_cnn.which_device(args.device)
    print(f'Selected device: {device}', flush=True)
    if args.pin_memory is False and device.type == 'cuda':
        args.pin_memory = True

    with open(run_dir / 'config.json', 'w') as f:
        json.dump(vars(args), f, indent=2)
    with open(run_dir / 'preprocessing_config.json', 'w') as f:
        json.dump({**payload['config'], 'metadata': payload['metadata']}, f, indent=2)
    with open(run_dir / 'model_config.json', 'w') as f:
        json.dump({
            'model': args.model,
            'base_channels': args.base_channels,
            'num_blocks': args.num_blocks,
            'out_dim': args.out_dim,
            'dropout': args.dropout,
            'norm': args.norm,
            'pooling': args.pooling,
            'input_shape': ds.input_shape,
            'parameters': train_ale_cnn.count_parameters(brain_encoder),
        }, f, indent=2)

    print('
== ALE Dataset ==', flush=True)
    print(f'  mode={args.mode} cache={args.cache_file}', flush=True)
    print(f'  volume shape={ds.input_shape} aligned papers={len(ds):,}', flush=True)
    print(f'  split train={len(train_ds):,} val={len(val_ds):,} test={len(test_ds):,}', flush=True)
    print('
== Model ==', flush=True)
    print(f'  {args.model} brain params={train_ale_cnn.count_parameters(brain_encoder):,}', flush=True)
    print(f'  text params={train_ale_cnn.count_parameters(text_proj):,} train_text={not args.freeze_text_proj}', flush=True)
    print(f'  device={device} amp={args.amp and device.type == "cuda"} batch={args.batch_size}', flush=True)

    trainer = train_ale_cnn.ALETrainer(brain_encoder, text_proj, args, device)
    trainer.fit(train_ds, val_ds)
    trainer.restore_best()

    all_metrics = {}
    eval_payloads = {}
    for split_name, split_ds in [('val', val_ds), ('test', test_ds)]:
        metrics, brain_emb, text_emb = trainer.evaluate(split_ds)
        all_metrics[split_name] = metrics
        eval_payloads[split_name] = (metrics, brain_emb, text_emb)
        print(
            f"
{split_name.upper()} n={len(split_ds):,} "
            f"paper_recall_curve_auc={metrics['paper_recall_curve_auc']:.4f} "
            f"r@1={metrics['mean_recall@1']:.4f} "
            f"r@5={metrics['mean_recall@5']:.4f} "
            f"r@10={metrics['mean_recall@10']:.4f} "
            f"r@50={metrics['mean_recall@50']:.4f} "
            f"MRR={metrics['mean_mrr']:.4f} "
            f"median_rank={metrics['mean_median_rank']:.1f} "
            f"random_r@10={metrics['mean_random_recall@10']:.4f}",
            flush=True,
        )

    with open(run_dir / 'training_history.json', 'w') as f:
        json.dump(trainer.history, f, indent=2)
    with open(run_dir / 'eval_results.json', 'w') as f:
        json.dump(all_metrics, f, indent=2)

    curve_frames = {}
    for split_name, (_, brain_emb, text_emb) in eval_payloads.items():
        curve_df = train_ale_cnn.recall_curve_frame(text_emb, brain_emb)
        curve_frames[split_name] = curve_df
        curve_df.to_csv(run_dir / f'{split_name}_recall_curve.csv', index=False)
        with open(run_dir / f'{split_name}_recall_curve.json', 'w') as f:
            json.dump(train_ale_cnn.recall_curve_payload(text_emb, brain_emb), f, indent=2)

    test_metrics, test_brain, test_text = eval_payloads['test']
    cov = test_ds.covariate_frame()
    diag_df = train_ale_cnn.retrieval_diagnostics(test_text, test_brain, cov)
    diag_df.to_csv(run_dir / 'test_retrieval_diagnostics.csv', index=False)
    train_ale_cnn.save_embedding_correlations(run_dir, test_brain, cov)
    if args.save_plots:
        train_ale_cnn.save_plots(run_dir, trainer.history, curve_frames['test'], diag_df if args.umap else None, test_brain)
    train_ale_cnn.append_comparison_row(args, payload, test_metrics, trainer, ds.input_shape)

    if os.path.exists(local_cache_file) and (not os.path.exists(drive_cache_file) or not local_cache_existed):
        print(f'Copying local packed cache back to Drive: {local_cache_file} -> {drive_cache_file}', flush=True)
        shutil.copy2(local_cache_file, drive_cache_file)

    run_info = {
        'mode': mode,
        'model': model,
        'drive_run_dir': drive_run_dir,
        'local_cache_file': local_cache_file,
        'drive_cache_file': drive_cache_file,
        'best_checkpoint': f'{drive_run_dir}/checkpoints/best_ale_cnn.pt',
        'last_checkpoint': f'{drive_run_dir}/checkpoints/last_ale_cnn.pt',
        'eval_results': f'{drive_run_dir}/eval_results.json',
        'training_history': f'{drive_run_dir}/training_history.json',
        'comparison_row': f'{drive_run_dir}/comparison_row.json',
    }
    COMPLETED_RUNS.append(run_info)
    with open(f'{drive_run_dir}/colab_run_info.json', 'w') as f:
        json.dump(run_info, f, indent=2)
    print(json.dumps(run_info, indent=2), flush=True)
    return run_info


## 7. Cell 1 - Run DiFuMo-Compatible ALE3DCNN

In [ ]:
difumo_run = run_ale_experiment(
    mode='difumo_compatible',
    model='ale_3dcnn',
    run_name=RUN_NAME_OR_TIMESTAMP,
)

## 8. Cell 2 - Run Atlas-Free ALE3DCNN

In [ ]:
atlas_free_run = run_ale_experiment(
    mode='atlas_free',
    model='ale_3dcnn',
    run_name=RUN_NAME_OR_TIMESTAMP,
)

## 9. Results Saved

In [ ]:
import pandas as pd
rows = COMPLETED_RUNS
if not rows:
    rows = [x for x in [globals().get('difumo_run'), globals().get('atlas_free_run')] if x]
pd.DataFrame(rows)

## 10. Comparison Notebook

The comparison CSV is saved at `DRIVE_ROOT/runs/ale_model_comparison.csv`. Run `experiments/compare_ale_models.ipynb` after both cells finish to make side-by-side tables and plots. If you want the comparison notebook to read Drive automatically, set `COMPARISON_FILE` there to the same path printed above.